In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum, avg, 
    round as _round, min as _min, max as _max, 
    window, desc
)

spark = (SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [11]:

df = spark.read.json("data/transactions_10k.jsonl")
print(f"Liczba rekordów: {df.count()}")

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
df.printSchema()
df.show(5, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows



In [23]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

print("Zadanie 2.1 - Liczba transakcji i suma przychodów per sklep")

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

Zadanie 2.1 - Liczba transakcji i suma przychodów per sklep
+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [13]:
print("Zadanie 2.2 - Statystyki per kategoria")
category_stats = (df.groupBy("category")
    .agg(
        _sum("amount").alias("suma_kwot"),
        _min("amount").alias("min_kwota"),
        _max("amount").alias("max_kwota")
    )
    .orderBy("category"))

category_stats.show()

Zadanie 2.2 - Statystyki per kategoria
+-----------+------------------+---------+---------+
|   category|         suma_kwot|min_kwota|max_kwota|
+-----------+------------------+---------+---------+
|elektronika|1520770.6899999995|      9.0|   9999.0|
|    książki| 851382.0799999991|      5.0|  9107.25|
|     odzież| 849877.5500000007|      5.0|  9696.63|
|    żywność| 789514.4300000003|      5.0|  6916.92|
+-----------+------------------+---------+---------+



In [25]:
from pyspark.sql.functions import window, col, count, sum as _sum, round as _round

print("Zadanie 3.1 - Liczba transakcji per godzina (tumbling 1h)")

from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

Zadanie 3.1 - Liczba transakcji per godzina (tumbling 1h)
+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [26]:
print("Zadanie 3.2 - Okna 30-minutowe per sklep")
thirty_min_windows = (df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN"
    )
    .orderBy("od", "store"))

thirty_min_windows.show(10, truncate=False)

Zadanie 3.2 - Okna 30-minutowe per sklep
+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
+-------------------+---------------

In [17]:
print("Zadanie 3.3 - Najwyższy przychód w Krakowie per godzina")
krakow_peak = (df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma_PLN"))
    .select(
        col("window.start").alias("godzina_start"),
        "suma_PLN"
    )
    .orderBy(desc("suma_PLN")))

krakow_peak.show(1)

Zadanie 3.3 - Najwyższy przychód w Krakowie per godzina
+-------------------+---------+
|      godzina_start| suma_PLN|
+-------------------+---------+
|2026-04-12 09:00:00|483309.86|
+-------------------+---------+
only showing top 1 row



In [28]:
print("Zadanie 4.1 - Okno 1h, krok 30 minut")
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes")) 
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

Zadanie 4.1 - Okno 1h, krok 30 minut
+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [29]:
print("Zadanie 4.2 - Porównaj tumbling vs sliding")

tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
tumbling_rows = df.groupBy(window("timestamp", "1 hour")).agg(count("tx_id")).count()
sliding_rows = df.groupBy(window("timestamp", "1 hour", "30 minutes")).agg(count("tx_id")).count()

print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Sliding ma więcej wierszy, ponieważ okna nakładają się na siebie. 
# Przy kroku (slide) 30 min i szerokości 1h, każde 30 minut startuje nowe okno, 
# więc każda transakcja (poza skrajnymi) wpada do dwóch okien jednocześnie.

Zadanie 4.2 - Porównaj tumbling vs sliding
Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [20]:
# 1. Ile transakcji jest w oknie 09:00–10:00?
# 4661.

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
# groupBy("store") agreguje wszystkie dane historyczne dla sklepu w jedną całość. 
# groupBy(window(...), "store") dzieli te dane na przedziały czasowe, pozwalając analizować dynamikę sprzedaży w czasie.

# 3. W oknie sliding 1h/30min - ile okien zawiera transakcje z godziny 09:30?
# Zawierają ją 2 okna 09:00 - 10:00 oraz 09:30 - 10:30.

In [31]:
print("Praca domowa")

# 1
gdansk_low_avg = (df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_PLN"))
    .orderBy("srednia_PLN"))
print("Gdańsk - najniższa średnia:")
gdansk_low_avg.show(1)

# 2
cat_09_00 = (df.filter((col("timestamp") >= "2024-03-04 09:00:00") & (col("timestamp") < "2024-03-04 09:30:00"))
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx")))
print("Transakcje 09:00-09:30 per kategoria:")
cat_09_00.show()

# 3
peak_15min = (df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx")))
print("Szczytowa ćwierćgodzina:")
peak_15min.show(1, truncate=False)

Praca domowa
Gdańsk - najniższa średnia:
+--------------------+-----------+
|              window|srednia_PLN|
+--------------------+-----------+
|{2026-04-12 08:00...|     395.01|
+--------------------+-----------+
only showing top 1 row

Transakcje 09:00-09:30 per kategoria:
+--------+---------+
|category|liczba_tx|
+--------+---------+
+--------+---------+

Szczytowa ćwierćgodzina:
+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+
only showing top 1 row

